# Sub-Saharan African bat occurrence records with taxonomy, locality, and specimen metadata Exploration with `mlcroissant`
This notebook follows a step-by-step workflow for loading, inspecting, and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.z4v4-15bp/fair2.json`

### Important note on `@id`
In Croissant, dataset entities such as record sets, fields, file objects, and columns are identified by their `@id` values. This notebook is written to inspect and use those identifiers dynamically whenever possible.

Because this dataset package summary indicates that `recordSet` is currently empty, the notebook includes robust metadata inspection steps and conditional record-loading code that will work if record sets are present in the live Croissant document.

In [ ]:
# Ensure `mlcroissant` and common analysis libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata from the Croissant schema URL and inspect the top-level dataset information.

This section deliberately accesses `dataset.metadata` as a single object and avoids treating it like a dictionary-like container directly.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.z4v4-15bp/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

# Convert metadata to a standard JSON-serializable structure for inspection
metadata_json = metadata_obj.to_json()

print('Dataset loaded successfully.')
print('\nName:')
print(metadata_json.get('name'))
print('\nDescription:')
print(metadata_json.get('description'))
print('\nIdentifier:')
print(metadata_json.get('identifier'))
print('\nVersion:')
print(metadata_json.get('version'))
print('\nLicense:')
print(metadata_json.get('license'))

In [ ]:
# A compact metadata summary for quick reference
summary_fields = [
    'name', 'identifier', 'version', 'datePublished', 'temporalCoverage',
    'spatialCoverage', 'license', 'citeAs'
]

summary_df = pd.DataFrame(
    [(field, metadata_json.get(field)) for field in summary_fields],
    columns=['metadata_field', 'value']
)
summary_df

## 2. Data Overview
Review the dataset structure, including available record sets and their `@id` values.

Since Croissant datasets can vary in how much structure they expose, we inspect the raw JSON-LD metadata and summarize the available entities. All record sets and fields are referenced by `@id`.

In [ ]:
# Inspect top-level keys in the metadata JSON
top_level_keys = sorted(metadata_json.keys())
print('Top-level metadata keys:')
print(top_level_keys)

In [ ]:
# Extract record set information from the metadata JSON
record_sets_json = metadata_json.get('recordSet', []) or []

record_set_rows = []
for rs in record_sets_json:
    record_set_rows.append({
        'record_set_id': rs.get('@id'),
        'name': rs.get('name'),
        'description': rs.get('description'),
        'field_count': len(rs.get('field', []) or [])
    })

record_sets_df = pd.DataFrame(record_set_rows)
print(f'Number of record sets declared in metadata: {len(record_set_rows)}')
record_sets_df

In [ ]:
# Build a field-level overview table using @id values
field_rows = []

for rs in record_sets_json:
    rs_id = rs.get('@id')
    rs_name = rs.get('name')
    for field in rs.get('field', []) or []:
        source_info = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        field_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'field_id': field.get('@id'),
            'field_name': field.get('name'),
            'description': field.get('description'),
            'data_type': field.get('dataType'),
            'source_file_object_id': source_info.get('fileObject'),
            'source_extract_column_id': source_info.get('extract', {}).get('column') if isinstance(source_info.get('extract'), dict) else None
        })

fields_df = pd.DataFrame(field_rows)
print(f'Number of fields declared across record sets: {len(field_rows)}')
fields_df.head(20)

The next cell prints a sample of records from a selected record set using its `@id`. If no record sets are present, it reports that clearly rather than failing.

In [ ]:
# Select a record set dynamically by @id, if one exists
record_set_ids = [rs.get('@id') for rs in record_sets_json if rs.get('@id')]
selected_record_set_id = record_set_ids[0] if record_set_ids else None

print('Available record set @id values:')
print(record_set_ids)
print('\nSelected record set @id:')
print(selected_record_set_id)

if selected_record_set_id is not None:
    print('\nSample records:')
    for i, record in enumerate(dataset.records(record_set=selected_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print('No record sets were declared in the Croissant metadata, so no records can be previewed from a record set.')

## 3. Data Extraction
Load data from each available record set into pandas DataFrames.

This section uses record set `@id` values discovered from the metadata. If the dataset exposes no record sets, the notebook will still remain useful for metadata exploration.

In [ ]:
# Extract all available record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(records)} records from record set: {record_set_id}')
    except Exception as exc:
        print(f'Could not load records from {record_set_id}: {exc}')

print('\nLoaded DataFrame keys:')
print(list(dataframes.keys()))

In [ ]:
# Inspect columns and preview the first loaded record set, if available
if dataframes:
    first_record_set_id = next(iter(dataframes))
    first_df = dataframes[first_record_set_id]
    print(f'First loaded record set @id: {first_record_set_id}')
    print('\nColumns:')
    print(first_df.columns.tolist())
    display(first_df.head())
else:
    print('No DataFrames were created because no loadable record sets were available.')

## 4. Exploratory Data Analysis (EDA)
Apply generic exploratory analysis that adapts to the available schema.

Because Croissant datasets can differ widely, the code below:
- identifies numeric and categorical columns dynamically,
- filters numeric values using a quantile-based threshold,
- normalizes a selected numeric field,
- groups by a categorical field when available,
- and reports missingness.

All selections are based on discovered record set and field/column names from the loaded data.

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    print(f'Using record set for EDA: {record_set_id}')
    print(f'DataFrame shape: {df.shape}')
    
    # Missingness overview
    missingness_df = (
        df.isna()
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    missingness_df.columns = ['column', 'missing_fraction']
    display(missingness_df.head(15))
else:
    print('EDA on records is skipped because no record-level DataFrame is available.')

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()

    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    categorical_columns = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

    print('Numeric columns:')
    print(numeric_columns)
    print('\nCategorical columns:')
    print(categorical_columns[:20])

    if numeric_columns:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].dropna().quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold].copy()

        print(f"\nSelected numeric field: {numeric_field}")
        print(f"Threshold (75th percentile): {threshold}")
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)}")
        display(filtered_df.head())

        std_value = filtered_df[numeric_field].std()
        if pd.notna(std_value) and std_value != 0:
            normalized_col = f'{numeric_field}_normalized'
            filtered_df[normalized_col] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
                std_value
            )
            print(f'Normalized field created: {normalized_col}')
            display(filtered_df[[numeric_field, normalized_col]].head())
        else:
            print('Normalization skipped because the filtered numeric field has zero or undefined standard deviation.')

        group_candidates = [c for c in categorical_columns if c in df.columns]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).reset_index()
            print(f'Grouped mean of {numeric_field} by {group_field}:')
            display(grouped_df.head(10))
        else:
            print('No categorical column available for grouping.')
    else:
        print('No numeric columns were detected, so numeric filtering and normalization were skipped.')
else:
    print('No DataFrame is available for adaptive EDA.')

For this dataset, geographic fields such as latitude and longitude may be especially relevant. The next cell looks for coordinate-like columns by name and summarizes them if present.

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()

    lower_name_map = {col.lower(): col for col in df.columns}
    latitude_candidates = [col for col in df.columns if 'lat' in col.lower()]
    longitude_candidates = [col for col in df.columns if 'lon' in col.lower() or 'lng' in col.lower()]

    print('Latitude-like columns:', latitude_candidates)
    print('Longitude-like columns:', longitude_candidates)

    if latitude_candidates and longitude_candidates:
        lat_col = latitude_candidates[0]
        lon_col = longitude_candidates[0]
        geo_df = df[[lat_col, lon_col]].copy()
        geo_df[lat_col] = pd.to_numeric(geo_df[lat_col], errors='coerce')
        geo_df[lon_col] = pd.to_numeric(geo_df[lon_col], errors='coerce')
        print(f'Using coordinate columns: {lat_col}, {lon_col}')
        display(geo_df.describe())
    else:
        print('No obvious coordinate columns were found by column name pattern matching.')
else:
    print('No DataFrame available for geographic summary.')

## 5. Visualization
Visualize distributions and relationships for whichever columns are available.

The visualizations below are adaptive:
- histogram for the first numeric column,
- bar plot for the most common values in the first categorical column,
- scatter plot for latitude/longitude-like columns if present.

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    categorical_columns = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

    if numeric_columns:
        numeric_field = numeric_columns[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()
    else:
        print('Skipping numeric histogram because no numeric columns are available.')

    if categorical_columns:
        cat_field = categorical_columns[0]
        top_counts = df[cat_field].astype(str).value_counts().head(10)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=top_counts.index, y=top_counts.values)
        plt.title(f'Top categories in {cat_field}')
        plt.xlabel(cat_field)
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print('Skipping categorical bar plot because no categorical columns are available.')
else:
    print('Visualization skipped because no record-level DataFrame is available.')

In [ ]:
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()

    latitude_candidates = [col for col in df.columns if 'lat' in col.lower()]
    longitude_candidates = [col for col in df.columns if 'lon' in col.lower() or 'lng' in col.lower()]

    if latitude_candidates and longitude_candidates:
        lat_col = latitude_candidates[0]
        lon_col = longitude_candidates[0]

        plot_df = df[[lat_col, lon_col]].copy()
        plot_df[lat_col] = pd.to_numeric(plot_df[lat_col], errors='coerce')
        plot_df[lon_col] = pd.to_numeric(plot_df[lon_col], errors='coerce')
        plot_df = plot_df.dropna().head(5000)

        if not plot_df.empty:
            plt.figure(figsize=(7, 6))
            sns.scatterplot(data=plot_df, x=lon_col, y=lat_col, s=20, alpha=0.6)
            plt.title(f'Spatial distribution using {lon_col} and {lat_col}')
            plt.xlabel(lon_col)
            plt.ylabel(lat_col)
            plt.tight_layout()
            plt.show()
        else:
            print('Coordinate columns were found, but no valid numeric coordinate pairs were available to plot.')
    else:
        print('Skipping coordinate scatter plot because latitude/longitude-like columns were not found.')
else:
    print('No DataFrame available for spatial visualization.')

## 6. Conclusion
This notebook demonstrated how to:

- load a dataset from a Croissant schema URL with `mlcroissant`,
- inspect the dataset metadata safely via `dataset.metadata` as a single object,
- discover record sets and fields using their `@id` values,
- load records dynamically into pandas DataFrames,
- perform adaptive EDA based on available numeric and categorical columns,
- and visualize basic distributions and possible spatial patterns.

### Observation for this dataset
The provided dataset summary indicates that `recordSet` is empty in the package metadata snapshot. If that remains true in the live Croissant document, the dataset is currently best explored at the metadata level rather than through tabular record extraction. If record sets are added or exposed later, the extraction and EDA sections of this notebook will work with minimal or no changes because they are written dynamically around record set `@id` values.

### Suggested next steps
- Inspect the live JSON-LD directly if you need to validate whether record sets are absent or simply not included in the package summary.
- If record sets become available, update the selected `record_set_id` to the desired `@id` for deeper domain-specific analysis.
- Add species-, country-, or collection-based plots once the corresponding field `@id` values are available in the metadata.